# Laboratorio 7

## Integrantes 

- Sergio Orellana 221122
- Rodrigo Mansilla 22611
- Ricardo Chuy 221007

## Contexto

Una empresa de ciberseguridad opera un sistema de deteccion de intrusiones en redes corporativas. El sistema monitorea trafico en tiempo real y debe decidir entre cuatro acciones: **ignorar, alertar, bloquear temporalmente o aislar el segmento afectado**. El estado es un vector numerico continuo de 256 dimensiones y los eventos criticos son extremadamente raros.

En este notebook desarrollo **Task 1 y Task 2 completas**, conservando las preguntas del laboratorio y respondiendo en primera persona con argumentacion tecnica.


# Task 1 (Entrega Parcial)

## Task 1.1

### El equipo de ingenieria propone usar DQN estandar para este sistema. Antes de implementar cualquier cosa, usted debe hacer un analisis critico de esa propuesta.


### a. El estado del sistema es un vector de 256 dimensiones con valores continuos. Argumenten si DQN es arquitecturalmente apropiado para este tipo de estado. ¿Que tipo de red usarian como aproximador de $Q(s,a;\mathbf{w})$ y por que, considerando que el estado no es una imagen sino un vector numerico?

_Respuesta:_

Considero que **DQN es arquitecturalmente apropiado** porque el espacio de estados es continuo y de alta dimension, mientras que el espacio de acciones es discreto. En este caso no usaria una red convolucional, ya que las 256 variables no forman una imagen y no existe una estructura espacial bidimensional que justifique filtros convolucionales.

Usaria una **red neuronal completamente conectada (MLP)**:

$$
\mathbf{s}\in\mathbb{R}^{256}
\rightarrow Dense \rightarrow ReLU \rightarrow Dense \rightarrow ReLU \rightarrow \mathbb{R}^{4}
$$

La salida contendria:

$$
Q(s,\text{ignorar}),\;Q(s,\text{alertar}),\;Q(s,\text{bloquear}),\;Q(s,\text{aislar})
$$

Por consiguiente, una sola evaluacion de la red estima simultaneamente el valor de las cuatro acciones. Ademas, una MLP puede aprender interacciones no lineales entre metricas de trafico, comportamiento de usuarios y logs recientes. Antes del entrenamiento normalizaria las 256 variables para evitar que diferencias de escala dificulten la optimizacion.

Por lo tanto, DQN es una opcion razonable siempre que el vector actual contenga suficiente informacion para aproximar el estado. Si existieran dependencias temporales que no estan resumidas en esas 256 variables, consideraria incorporar memoria mediante una arquitectura recurrente.


### b. El espacio de acciones tiene 4 opciones discretas. Argumenten si DQN o alguna de sus variantes es apropiado para este espacio de acciones, o si seria necesario un algoritmo diferente.

_Respuesta:_

Considero que DQN es especialmente apropiado porque las cuatro acciones son **discretas y finitas**. La accion greedy puede seleccionarse mediante:

$$
a^*=\arg\max_a Q(s,a;\mathbf{w})
$$

No necesito un algoritmo orientado a acciones continuas, como DDPG, TD3 o SAC. Esos metodos tendrian mas sentido si la accion fuera un nivel continuo de filtrado o bloqueo.

Asimismo, variantes como **Double DQN**, **Dueling DQN** y DQN con **Prioritized Experience Replay** siguen siendo compatibles con este espacio. Por lo tanto, la naturaleza de las acciones favorece el uso de DQN y sus extensiones.


### c. En el dominio de deteccion de intrusiones, los eventos criticos ocurren en menos del 0.1% de los pasos de tiempo. Argumenten formalmente que consecuencia tiene esa rareza sobre el buffer de Experience Replay y sobre la distribucion de muestreo uniforme. ¿Que variante de DQN resolveria este problema especifico y como?

_Respuesta:_

Si una intrusión real ocurre con probabilidad aproximada $p=0.001$, entonces en un buffer de $N$ transiciones espero:

$$
E[N_{crit}]=Np
$$

Con una capacidad de 50,000 transiciones:

$$
E[N_{crit}]=50{,}000(0.001)=50
$$

Si tomo minibatches uniformes de 64 transiciones, el numero esperado de eventos criticos por minibatch es:

$$
64(0.001)=0.064
$$

La probabilidad de obtener al menos una experiencia critica es:

$$
P(X\geq1)=1-(1-0.001)^{64}\approx0.062
$$

Por consiguiente, alrededor del 93.8% de los minibatches no contendria ninguna intrusión si la tasa fuera exactamente 0.1%, y la situacion seria peor si la frecuencia real fuera menor.

Esto hace que el replay uniforme entrene principalmente con trafico normal. Por ello usaria **Prioritized Experience Replay (PER)**, donde:

$$
p_i=(|\delta_i|+\epsilon_p)^\alpha
$$

$$
P(i)=\frac{p_i}{\sum_k p_k}
$$

Las transiciones raras que producen errores TD grandes se reutilizan con mayor frecuencia. Ademas, emplearia pesos de importancia:

$$
w_i=\left(\frac{1}{N P(i)}\right)^\beta
$$

para reducir el sesgo introducido por el muestreo no uniforme. Por lo tanto, considero que **Double DQN + PER** es una alternativa mas adecuada que DQN estandar con replay uniforme.


### d. El sistema actual basado en reglas tiene una tasa de falsos positivos del 8%. Si disenaran la funcion de recompensa de DQN para minimizar falsos positivos unicamente, ¿que comportamiento indeseable podria aprender el agente? Disenen una funcion de recompensa con al menos tres componentes que capture el objetivo real del sistema, justificando la magnitud relativa de cada uno.

_Respuesta:_

Si solamente penalizo falsos positivos, el agente podria aprender a **ignorar casi todo** el trafico sospechoso. Eso reduciria bloqueos incorrectos, pero aumentaria falsos negativos e intrusiones no contenidas.

Propongo:

$$
R_t=100I_{TP}-150I_{FN}-40I_{FP}+10I_{alerta\ util}-5C(a_t)
$$

- $+100$ por bloquear o aislar correctamente una intrusion real.
- $-150$ por ignorar una intrusion real.
- $-40$ por bloquear trafico legitimo.
- $+10$ por una alerta util que evita una respuesta agresiva prematura.
- $-5$ por acciones operativamente costosas innecesarias.

Asigno la mayor penalizacion al falso negativo porque puede comprometer sistemas completos. Sin embargo, tambien penalizo de manera relevante los falsos positivos porque interrumpen operaciones legitimas. Por lo tanto, la recompensa balancea **seguridad, continuidad operativa y costo de respuesta**.


## Task 1.2

### El equipo propone usar Double DQN en lugar de DQN estandar.


### a. Expliquen formalmente el problema de sobreestimacion que Double DQN resuelve. En el contexto especifico de deteccion de intrusiones, ¿que consecuencia operacional tendria que el agente sobreestime sistematicamente el valor de la accion "bloquear"? ¿Y de la accion "ignorar"?

_Respuesta:_

Si las estimaciones contienen ruido:

$$
\hat Q(s,a)=Q^*(s,a)+\epsilon_a
$$

el operador maximo tiende a seleccionar acciones cuyo error es accidentalmente positivo. Por ello:

$$
E[\max_a \hat Q(s,a)]\geq \max_a E[\hat Q(s,a)]
$$

DQN puede sobreestimar los targets. Double DQN reduce este sesgo al separar seleccion y evaluacion.

Si el agente sobreestima **bloquear**, podria bloquear trafico legitimo con excesiva frecuencia y aumentar falsos positivos. Si sobreestima **ignorar**, podria descartar actividad realmente maliciosa y aumentar falsos negativos. En ambos casos, el sesgo de valor se convierte en un riesgo operacional real.


### b. Escriban la expresion del target de DQN estandar y la del target de Double DQN, identificando con precision que parametros hacen que en cada caso. ¿Cual es el unico cambio de implementacion entre ambos?

_Respuesta:_

En DQN estandar:

$$
y_t^{DQN}=r_t+\gamma(1-d_t)\max_{a'}Q(s_{t+1},a';\mathbf{w}^{-})
$$

En Double DQN:

$$
a^*=\arg\max_{a'}Q(s_{t+1},a';\mathbf{w})
$$

$$
y_t^{DDQN}=r_t+\gamma(1-d_t)Q(s_{t+1},a^*;\mathbf{w}^{-})
$$

La red online $\mathbf{w}$ selecciona la accion y la Target Network $\mathbf{w}^{-}$ la evalua. El cambio fundamental esta solamente en el calculo del target; la arquitectura, el buffer y la actualizacion periodica de la Target Network pueden mantenerse iguales.


### c. Argumenten si en este dominio especifico la sobreestimacion es un problema mas grave en ciertos tipos de estados que en otros. Consideren estados con alta ambiguedad (trafico borderline) versus estados claramente maliciosos.

_Respuesta:_

Considero que el problema es mas grave en **estados ambiguos**. En ellos puede ocurrir:

$$
Q^*(s,alertar)\approx Q^*(s,bloquear)\approx Q^*(s,ignorar)
$$

Cuando las diferencias reales son pequenas, el ruido puede cambiar facilmente cual accion aparece como maxima. En un estado claramente malicioso esperaria una separacion mayor, por ejemplo:

$$
Q^*(s,bloquear)\gg Q^*(s,ignorar)
$$

Por consiguiente, la sobreestimacion es especialmente peligrosa en regiones borderline, donde una pequena diferencia puede convertir una decision de alerta en un bloqueo innecesario o una decision de ignorar en un falso negativo.


## Registro del uso de IA generativa

**Prompt utilizado**

> Quiero que me ayudes a entender cada uno de los incisos y como los plantearias para resolverlos, cabe mencionas que no quiero que los resuelvas si no que me proporciones una guía para plantear mi propia respuesta pero basada con lo que tu me proporcionas y no con lo que tu generes, ademas quiero que me ayudes a entender el porque de cada respuesta y que me ayudes a entender la logica de cada una de ellas. 

**Por que funciono**

El prompt fue efectivo porque me proporciono una guia clara para abordar cada inciso, enfocandome en la logica y razonamiento detras de cada respuesta. Esto me permitio entender mejor los conceptos y como aplicarlos a mi propio analisis, sin depender de respuestas generadas directamente.


## Referencias

- GeeksforGeeks. (2026, July 7). Deep Q-learning in reinforcement learning. https://www.geeksforgeeks.org/deep-learning/deep-q-learning/

- GeeksforGeeks. (2025, July 23). Understanding prioritized experience replay. https://www.geeksforgeeks.org/machine-learning/understanding-prioritized-experience-replay/

- GeeksforGeeks. (2025, July 23). Sparse rewards in reinforcement learning. https://www.geeksforgeeks.org/machine-learning/sparse-rewards-in-reinforcement-learning/

- GeeksforGeeks. (2026, July 15). Deep reinforcement learning. https://www.geeksforgeeks.org/artificial-intelligence/deep-reinforcement-learning/

- GeeksforGeeks. (2026, April 30). Reinforcement learning. https://www.geeksforgeeks.org/machine-learning/what-is-reinforcement-learning/
